# Advanced Pickling — Tutorial-Style Problems with Solutions

This notebook continues beyond basic `pickle.dumps`, `pickle.loads`, `pickle.dump`, and `pickle.load`.

Instead of presenting each exercise as a single block, we will work through the problems in small logical steps.

The overall pattern will usually be:

1. **Understand the situation**
2. **Make a prediction**
3. **Run a small experiment**
4. **Inspect the result**
5. **Explain why it happened**
6. **Solve the full problem**
7. **Extract a best practice**


## Important Security Warning

Pickle is **not a secure data format**.

Unpickling can cause Python to import globals and reconstruct objects in ways that may execute code.

Therefore:

> **Never unpickle data that came from an untrusted or unauthenticated source.**

In this notebook, all examples are intentionally local and controlled.

We will study pickle's behavior, but we will not create a harmful command-execution payload.


## Imports

We will use only Python's standard library.


In [ ]:
from __future__ import annotations

import builtins
import copyreg
import gzip
import hashlib
import hmac
import io
import os
import pickle
import pickletools
import statistics
import tempfile
import timeit

from dataclasses import dataclass, field
from pathlib import Path
from pprint import pprint
from typing import Any


Before we start, let's see which pickle protocol is the highest one supported by this Python runtime.


In [ ]:
pickle.HIGHEST_PROTOCOL

# Problem 1 — Understanding Pickle's Memo

One of pickle's most important ideas is the **memo**.

The memo allows pickle to remember objects it has already serialized.

This matters when:

- the same object is referenced multiple times;
- an object graph contains cycles;
- object identity relationships matter inside the serialized graph.

Let's investigate this slowly.


## Step 1 — Create one shared object

We will create one list and store two references to that same list.


In [ ]:
shared_numbers = [10, 20, 30]

container = {
    "first": shared_numbers,
    "second": shared_numbers,
}


Before pickling anything, let's verify that both dictionary values point to the exact same list.


In [ ]:
container["first"] is container["second"]

That should be `True`.

Now let's serialize the whole dictionary as **one object graph**.


In [ ]:
serialized = pickle.dumps(
    container,
    protocol=pickle.HIGHEST_PROTOCOL
)


And now let's deserialize it.


In [ ]:
restored = pickle.loads(serialized)


## Step 2 — Compare equality

The restored dictionary should contain equivalent values.


In [ ]:
restored == container

## Step 3 — Compare identity with the original

The restored list is newly created, so it should not be the original list object.


In [ ]:
restored["first"] is shared_numbers

That should be `False`.

But now comes the more interesting question.

Inside the **restored** graph, do the two keys still refer to the same reconstructed list?


In [ ]:
restored["first"] is restored["second"]

That should be `True`.

This is one of the most important distinctions when working with pickle:

- identity relative to the **old process objects** is lost;
- identity relationships **inside the serialized object graph** can be preserved.


## Step 4 — Prove the alias is real

If both keys point to the same list, changing the list through one key should be visible through the other key.


In [ ]:
restored["first"].append(40)

print(restored["first"])
print(restored["second"])


## Full Solution

The correct interpretation is:

- `restored == container` → usually `False` now only because we mutated `restored`; before mutation it was `True`;
- `restored["first"] is shared_numbers` → `False`;
- `restored["first"] is restored["second"]` → `True`.

Pickle preserves the relative alias relationship because both references were serialized by the same `Pickler` memo.


## Best Practice

If shared identity between several objects matters, serialize the connected objects together whenever possible.


# Problem 2 — When Separate Pickles Break Relationships

The previous problem worked because we serialized one connected object graph.

Now let's see what happens if we serialize related objects separately.


## Step 1 — Build a relationship

`profile` will be used both directly and inside another dictionary.


In [ ]:
profile = {
    "name": "Ada",
    "language": "Python",
}

session = {
    "session_id": 101,
    "profile": profile,
}


Let's verify the relationship before serialization.


In [ ]:
session["profile"] is profile

## Step 2 — Serialize them independently

This creates two independent serialization operations.


In [ ]:
profile_bytes = pickle.dumps(
    profile,
    protocol=pickle.HIGHEST_PROTOCOL
)

session_bytes = pickle.dumps(
    session,
    protocol=pickle.HIGHEST_PROTOCOL
)


Now restore both.


In [ ]:
profile_restored = pickle.loads(profile_bytes)
session_restored = pickle.loads(session_bytes)


## Step 3 — Compare the restored values


In [ ]:
session_restored["profile"] == profile_restored

They should be equal in value.

But are they the same object?


In [ ]:
session_restored["profile"] is profile_restored

The answer should be `False`.

Why?

Because each call to `pickle.dumps` used a separate pickler and therefore a separate memo.


## Step 4 — Solve the design problem

Suppose the identity relationship matters.

Instead of storing two separate pickle blobs, serialize a parent object that contains everything.


In [ ]:
bundle = {
    "profile": profile,
    "session": session,
}

bundle_bytes = pickle.dumps(
    bundle,
    protocol=pickle.HIGHEST_PROTOCOL
)

bundle_restored = pickle.loads(bundle_bytes)


In [ ]:
bundle_restored["profile"] is bundle_restored["session"]["profile"]

## Solution Explanation

When the connected objects are serialized together, one memo sees the complete graph.

That allows pickle to reconstruct the shared reference correctly.


## Best Practice

Do not think of pickle as independently copying every nested value.

Think of it as serializing an **object graph**.


# Problem 3 — A Pickler Instance Can Preserve Memo State Across Multiple Dumps

There is a more subtle version of the previous problem.

Usually, calling `pickle.dump(...)` twice creates two different `Pickler` instances.

But what happens if we explicitly create **one `pickle.Pickler` object** and call its `.dump(...)` method multiple times?

Let's test it.


## Step 1 — Create two top-level objects that share a child


In [ ]:
shared_settings = {
    "theme": "dark",
    "language": "en",
}

user_a = {
    "name": "Alice",
    "settings": shared_settings,
}

user_b = {
    "name": "Bob",
    "settings": shared_settings,
}


Both users point to the same settings dictionary.


In [ ]:
user_a["settings"] is user_b["settings"]

## Step 2 — Use two convenience calls

First, we use `pickle.dump` twice.


In [ ]:
buffer_1 = io.BytesIO()

pickle.dump(
    user_a,
    buffer_1,
    protocol=pickle.HIGHEST_PROTOCOL
)

pickle.dump(
    user_b,
    buffer_1,
    protocol=pickle.HIGHEST_PROTOCOL
)


Now read the two objects back using two convenience `pickle.load` calls.


In [ ]:
buffer_1.seek(0)

loaded_a1 = pickle.load(buffer_1)
loaded_b1 = pickle.load(buffer_1)

loaded_a1["settings"] is loaded_b1["settings"]


That will normally be `False`.

Each convenience call creates a separate memo context.


## Step 3 — Reuse one Pickler

Now we will keep one `Pickler` object alive.


In [ ]:
buffer_2 = io.BytesIO()

pickler = pickle.Pickler(
    buffer_2,
    protocol=pickle.HIGHEST_PROTOCOL
)

pickler.dump(user_a)
pickler.dump(user_b)


To preserve the same relationship while reading, we also reuse one `Unpickler`.


In [ ]:
buffer_2.seek(0)

unpickler = pickle.Unpickler(buffer_2)

loaded_a2 = unpickler.load()
loaded_b2 = unpickler.load()

loaded_a2["settings"] is loaded_b2["settings"]


## Solution Explanation

The memo belongs to the `Pickler` instance.

If the same `Pickler` instance performs multiple `.dump()` operations, the memo can continue to recognize objects it has already seen.

The same idea applies to a reused `Unpickler`.


## Best Practice

This technique can be useful in specialized streaming formats.

However, it also couples records together.

If one record depends on memo state created by earlier records, you can no longer load it independently.

Use this design only when that coupling is intentional.


# Problem 4 — Cyclic References

Python objects can reference themselves.

Many simple serialization strategies fail on cycles because they recurse forever.

Pickle supports cyclic graphs.

Let's see how.


## Step 1 — Create a self-referencing list


In [ ]:
loop = ["start"]
loop.append(loop)


Let's inspect only the first element normally.


In [ ]:
loop[0]

The second element is the list itself.


In [ ]:
loop[1] is loop

## Step 2 — Pickle and restore


In [ ]:
loop_bytes = pickle.dumps(
    loop,
    protocol=pickle.HIGHEST_PROTOCOL
)

loop_restored = pickle.loads(loop_bytes)


## Step 3 — Verify the cycle


In [ ]:
loop_restored[1] is loop_restored

## Why This Works

Pickle records the list in its memo.

When it encounters the list again while serializing the second element, it can emit a reference to the already-known object instead of recursively serializing it forever.


## Mini Challenge

Create two dictionaries that point to each other.

For example:

- `a["other"] is b`
- `b["other"] is a`

Then serialize only `a`.

Will the restored graph still contain both dictionaries and the cycle?


## Solution


In [ ]:
a = {"name": "A"}
b = {"name": "B"}

a["other"] = b
b["other"] = a

a_restored = pickle.loads(
    pickle.dumps(
        a,
        protocol=pickle.HIGHEST_PROTOCOL
    )
)

print(a_restored["name"])
print(a_restored["other"]["name"])

a_restored["other"]["other"] is a_restored


The answer is `True`.

Serializing `a` is enough because `b` is reachable from `a`, and `a` is reachable again from `b`.


# Problem 5 — Protocols Are Part of Your Compatibility Strategy

Pickle has several protocol versions.

Higher protocols usually provide newer encodings and may be smaller or faster, but older Python runtimes may not understand them.

We will compare protocol outputs.


## Step 1 — Build a moderately rich object


In [ ]:
sample = {
    "name": "protocol-demo",
    "numbers": list(range(200)),
    "labels": [f"item-{i % 10}" for i in range(200)],
    "flags": [True, False] * 100,
}


## Step 2 — Serialize using every supported protocol


In [ ]:
protocol_sizes = {}

for protocol in range(pickle.HIGHEST_PROTOCOL + 1):
    data = pickle.dumps(
        sample,
        protocol=protocol
    )
    protocol_sizes[protocol] = len(data)

protocol_sizes


## Step 3 — Find the smallest result on this machine


In [ ]:
min(
    protocol_sizes.items(),
    key=lambda item: item[1]
)


Do not assume the highest protocol is always dramatically smaller.

Protocol choice is mostly about:

- available features;
- compatibility;
- performance;
- representation improvements.


## Step 4 — Inspect the pickle bytecode

The `pickletools` module can disassemble pickle data without reconstructing the object.


In [ ]:
latest = pickle.dumps(
    sample,
    protocol=pickle.HIGHEST_PROTOCOL
)

pickletools.dis(latest)


## Solution Explanation

A pickle is not just raw object memory.

It is a sequence of pickle opcodes interpreted by the unpickler.

Different protocols introduce or change available opcodes and encodings.


## Best Practice

For trusted persistence where all environments are current and compatible:

```python
protocol=pickle.HIGHEST_PROTOCOL
```

is a sensible default.

For shared systems, choose the newest protocol supported by the oldest Python runtime that must read the data.


# Problem 6 — Do Not Serialize Recomputable Cache State

Suppose a class keeps a cache that can be recreated from the real data.

Serializing the cache has several disadvantages:

- larger pickle size;
- stale cached values;
- tighter coupling to implementation details.

We will use `__getstate__` and `__setstate__`.


## Step 1 — Start with a class that contains a cache


In [ ]:
@dataclass
class Measurements:
    values: list[float]
    _cache: dict[str, float] = field(
        default_factory=dict,
        repr=False
    )

    def average(self) -> float:
        if "average" not in self._cache:
            self._cache["average"] = (
                sum(self.values) / len(self.values)
            )
        return self._cache["average"]


Let's populate the cache.


In [ ]:
m = Measurements([10, 20, 30])

m.average()
m._cache


If we pickle this class without customization, `_cache` will also be part of its instance state.

Let's improve that.


## Step 2 — Add custom state methods


In [ ]:
@dataclass
class BetterMeasurements:
    values: list[float]
    _cache: dict[str, float] = field(
        default_factory=dict,
        repr=False
    )

    def average(self) -> float:
        if "average" not in self._cache:
            self._cache["average"] = (
                sum(self.values) / len(self.values)
            )
        return self._cache["average"]

    def __getstate__(self):
        state = self.__dict__.copy()
        state.pop("_cache", None)
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._cache = {}


## Step 3 — Populate the cache before serializing


In [ ]:
m1 = BetterMeasurements([10, 20, 30])

print(m1.average())
print(m1._cache)


## Step 4 — Round-trip the object


In [ ]:
m2 = pickle.loads(
    pickle.dumps(
        m1,
        protocol=pickle.HIGHEST_PROTOCOL
    )
)

print(m2.values)
print(m2._cache)


The meaningful values survived, but the cache was rebuilt as empty.


## Step 5 — Prove the cache still works


In [ ]:
print(m2.average())
print(m2._cache)


## Best Practice

Persist source-of-truth state.

Avoid persisting:

- caches;
- locks;
- open files;
- open sockets;
- thread objects;
- temporary connections;
- state that can be cheaply recomputed.


# Problem 7 — Revalidate Invariants When Restoring State

A class can have rules that must always remain true.

For example, a warehouse stock count should never be negative.

If older or corrupted application state reaches `__setstate__`, the class should not blindly accept it.


## Step 1 — Define a class with an invariant


In [ ]:
@dataclass
class InventoryItem:
    sku: str
    quantity: int

    def __post_init__(self):
        self._validate()

    def _validate(self):
        if self.quantity < 0:
            raise ValueError(
                "quantity must be non-negative"
            )

    def __getstate__(self):
        return {
            "sku": self.sku,
            "quantity": self.quantity,
        }

    def __setstate__(self, state):
        self.sku = state["sku"]
        self.quantity = state["quantity"]
        self._validate()


## Step 2 — Confirm valid objects round-trip


In [ ]:
item = InventoryItem(
    sku="ABC-100",
    quantity=25
)

item2 = pickle.loads(
    pickle.dumps(
        item,
        protocol=pickle.HIGHEST_PROTOCOL
    )
)

item2


## Step 3 — Simulate invalid historical state

We will not build a malicious pickle.

Instead, we directly call `__setstate__` to test the class's restore logic.


In [ ]:
invalid_state = {
    "sku": "BROKEN-1",
    "quantity": -10,
}

obj = InventoryItem.__new__(InventoryItem)

try:
    obj.__setstate__(invalid_state)
except ValueError as exc:
    print(
        "Rejected invalid state:",
        exc
    )


## Solution Explanation

Serialization hooks are not only about saving space.

They are also an opportunity to restore class invariants correctly.


## Best Practice

Treat unpickling as object construction.

If your normal constructor validates state, your restoration path should also validate state.


# Problem 8 — Migrating Old Serialized State

Long-lived applications evolve.

A field that once existed may later be renamed or reorganized.

If you control the class, `__setstate__` can migrate old state into the new format.


## Scenario

Version 1 stored a user's name as:

```python
{
    "version": 1,
    "first_name": "...",
    "last_name": "..."
}
```

Version 2 stores:

```python
{
    "version": 2,
    "display_name": "..."
}
```

We want current code to understand both.


In [ ]:
class UserRecord:
    CURRENT_VERSION = 2

    def __init__(self, display_name):
        self.display_name = display_name

    def __getstate__(self):
        return {
            "version": self.CURRENT_VERSION,
            "display_name": self.display_name,
        }

    def __setstate__(self, state):
        version = state.get(
            "version",
            1
        )

        if version == 1:
            first = state["first_name"]
            last = state["last_name"]

            self.display_name = (
                f"{first} {last}"
            ).strip()

        elif version == 2:
            self.display_name = (
                state["display_name"]
            )

        else:
            raise ValueError(
                f"Unsupported state version: {version}"
            )

    def __repr__(self):
        return (
            f"UserRecord("
            f"display_name={self.display_name!r})"
        )


## Step 1 — Simulate restoring version 1 state


In [ ]:
legacy_state = {
    "version": 1,
    "first_name": "Grace",
    "last_name": "Hopper",
}

legacy_user = UserRecord.__new__(
    UserRecord
)

legacy_user.__setstate__(
    legacy_state
)

legacy_user


## Step 2 — Confirm version 2 still round-trips


In [ ]:
current_user = UserRecord(
    "Ada Lovelace"
)

current_user_restored = pickle.loads(
    pickle.dumps(
        current_user,
        protocol=pickle.HIGHEST_PROTOCOL
    )
)

current_user_restored


## Step 3 — Reject unknown future versions


In [ ]:
future_state = {
    "version": 999,
    "display_name": "Future User",
}

future_user = UserRecord.__new__(
    UserRecord
)

try:
    future_user.__setstate__(
        future_state
    )
except ValueError as exc:
    print(exc)


## Best Practice

If serialized state is expected to survive code changes, include an explicit state version.

Do not rely on accidental compatibility.


# Problem 9 — Custom Serialization for a Class You Do Not Want to Modify

Sometimes you want to control pickling without adding methods directly to the class.

Python's `copyreg` module lets you register custom reduction logic.


## Step 1 — Define a simple class


In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return (
            f"Point("
            f"x={self.x}, y={self.y})"
        )


## Step 2 — Define how to reduce and reconstruct the object

Our reducer will convert a `Point` into constructor information.


In [ ]:
def reduce_point(point):
    return (
        Point,
        (point.x, point.y)
    )


## Step 3 — Register the reducer


In [ ]:
copyreg.pickle(
    Point,
    reduce_point
)


## Step 4 — Test the round trip


In [ ]:
p1 = Point(10, 20)

p2 = pickle.loads(
    pickle.dumps(
        p1,
        protocol=pickle.HIGHEST_PROTOCOL
    )
)

print(p1)
print(p2)

print(
    p1 is p2,
    (p1.x, p1.y) == (p2.x, p2.y)
)


## Why This Is Useful

`copyreg` is helpful when:

- you do not want serialization code inside the class;
- the class comes from another module;
- you want centralized serialization rules.

However, the registered reconstruction callable still becomes part of pickle's object-reconstruction mechanism.

That is another reason untrusted pickle is dangerous.


# Problem 10 — Externalizing Large Objects with Persistent IDs

Suppose an application contains references to large binary resources.

We may want the pickle to store only a lightweight reference rather than embedding the entire resource.

Pickle supports this through:

- `Pickler.persistent_id`;
- `Unpickler.persistent_load`.


## Step 1 — Create a reference class

This class represents a key into an external blob store.


In [ ]:
@dataclass(frozen=True)
class AssetRef:
    key: str


## Step 2 — Create a custom Pickler

Whenever the pickler sees an `AssetRef`, it will store a persistent identifier instead of serializing the object normally.


In [ ]:
class AssetPickler(pickle.Pickler):
    def persistent_id(self, obj):
        if isinstance(
            obj,
            AssetRef
        ):
            return (
                "ASSET",
                obj.key
            )

        return None


## Step 3 — Create a custom Unpickler

The unpickler will resolve the persistent ID from an external dictionary.


In [ ]:
class AssetUnpickler(
    pickle.Unpickler
):
    def __init__(
        self,
        file,
        asset_store
    ):
        super().__init__(file)
        self.asset_store = asset_store

    def persistent_load(self, pid):
        kind, key = pid

        if kind != "ASSET":
            raise pickle.UnpicklingError(
                f"Unsupported persistent id: {pid!r}"
            )

        try:
            return self.asset_store[key]
        except KeyError as exc:
            raise pickle.UnpicklingError(
                f"Missing asset: {key!r}"
            ) from exc


## Step 4 — Create an external store


In [ ]:
asset_store = {
    "logo": b"binary-logo-data",
    "manual": b"binary-manual-data",
}


## Step 5 — Build an object containing references


In [ ]:
project = {
    "title": "Serialization Demo",
    "logo": AssetRef("logo"),
    "manual": AssetRef("manual"),
}


## Step 6 — Serialize with the custom Pickler


In [ ]:
buffer = io.BytesIO()

AssetPickler(
    buffer,
    protocol=pickle.HIGHEST_PROTOCOL
).dump(project)

len(buffer.getvalue())


## Step 7 — Restore using the external store


In [ ]:
buffer.seek(0)

project_restored = AssetUnpickler(
    buffer,
    asset_store
).load()

pprint(project_restored)


Notice that the restored values are the actual bytes from `asset_store`, not `AssetRef` instances.


## Best Practice

Persistent IDs are useful when the pickle is only one part of a larger storage architecture.

But you must define what should happen when:

- an external key is missing;
- an asset changes;
- an asset is deleted;
- versions become incompatible.


# Problem 11 — Authenticate Before You Unpickle

Suppose two trusted components share a secret key.

We still want to detect accidental or malicious modification of the pickle bytes before loading them.

An HMAC can authenticate the payload.


## Step 1 — Understand the correct order

The safe order is:

1. receive bytes;
2. split authentication tag from payload;
3. compute the expected HMAC;
4. compare tags;
5. **only then** call `pickle.loads`.

Do not deserialize first and verify later.


In [ ]:
HMAC_SIZE = hashlib.sha256().digest_size


## Step 2 — Write the sealing function


In [ ]:
def seal_object(
    obj: Any,
    key: bytes
) -> bytes:
    payload = pickle.dumps(
        obj,
        protocol=pickle.HIGHEST_PROTOCOL
    )

    tag = hmac.new(
        key,
        payload,
        hashlib.sha256
    ).digest()

    return tag + payload


## Step 3 — Write the verification function


In [ ]:
def open_object(
    package: bytes,
    key: bytes
) -> Any:
    if len(package) < HMAC_SIZE:
        raise ValueError(
            "Package is too short"
        )

    received_tag = (
        package[:HMAC_SIZE]
    )

    payload = (
        package[HMAC_SIZE:]
    )

    expected_tag = hmac.new(
        key,
        payload,
        hashlib.sha256
    ).digest()

    if not hmac.compare_digest(
        received_tag,
        expected_tag
    ):
        raise ValueError(
            "Authentication failed"
        )

    return pickle.loads(payload)


## Step 4 — Test a valid package


In [ ]:
secret = b"training-secret"

message = {
    "job": "daily-backup",
    "attempt": 3,
}

package = seal_object(
    message,
    secret
)

open_object(
    package,
    secret
)


## Step 5 — Tamper with one byte


In [ ]:
tampered = bytearray(package)

tampered[-1] ^= 1


The modified package should be rejected before pickle gets a chance to interpret the payload.


In [ ]:
try:
    open_object(
        bytes(tampered),
        secret
    )
except ValueError as exc:
    print(
        "Rejected:",
        exc
    )


## Important Limitation

HMAC does **not** make pickle universally safe.

It only proves that the bytes came from someone who knew the secret key.

If the producer itself is malicious or compromised, it can create a malicious but correctly authenticated pickle.


# Problem 12 — Building a Very Small Restricted Unpickler

Another defensive technique is to limit which globals may be imported during unpickling.

This is useful as **defense in depth**.

It should not be treated as a perfect sandbox.


## Step 1 — Create a whitelist

We will allow only two built-in global constructors:

- `set`;
- `frozenset`.

Basic lists, dictionaries, strings, numbers, booleans, and `None` generally do not need global lookup.


In [ ]:
class TinyRestrictedUnpickler(
    pickle.Unpickler
):
    ALLOWED = {
        ("builtins", "set"): set,
        ("builtins", "frozenset"): frozenset,
    }

    def find_class(
        self,
        module,
        name
    ):
        key = (
            module,
            name
        )

        if key in self.ALLOWED:
            return self.ALLOWED[key]

        raise pickle.UnpicklingError(
            f"Forbidden global: "
            f"{module}.{name}"
        )


## Step 2 — Add a helper function


In [ ]:
def tiny_restricted_loads(
    data: bytes
):
    return TinyRestrictedUnpickler(
        io.BytesIO(data)
    ).load()


## Step 3 — Load ordinary data


In [ ]:
ordinary = {
    "name": "safe-shape",
    "values": [1, 2, 3],
    "tags": {"a", "b"},
}

ordinary_blob = pickle.dumps(
    ordinary,
    protocol=pickle.HIGHEST_PROTOCOL
)

tiny_restricted_loads(
    ordinary_blob
)


## Step 4 — Try a custom class

The restricted unpickler should reject it because it would require resolving a user-defined global.


In [ ]:
class ExampleClass:
    def __init__(self, value):
        self.value = value


example_blob = pickle.dumps(
    ExampleClass(99),
    protocol=pickle.HIGHEST_PROTOCOL
)

try:
    tiny_restricted_loads(
        example_blob
    )
except pickle.UnpicklingError as exc:
    print(
        "Rejected:",
        exc
    )


## Best Practice

A whitelist must stay very small.

Adding more reconstruction callables increases the attack surface.

When data is genuinely untrusted, use a safer serialization format instead.


# Problem 13 — Avoid Corrupting a Checkpoint During a Crash

Suppose we write a pickle directly to:

```text
checkpoint.pkl
```

If the process crashes halfway through the write, the file may become unusable.

A common solution is an **atomic replace** strategy.


## Step 1 — The strategy

We will:

1. create a temporary file in the same directory;
2. write the complete pickle;
3. flush Python's buffer;
4. call `os.fsync`;
5. replace the destination using `os.replace`.

The destination is replaced only after the temporary file is ready.


In [ ]:
def atomic_pickle_dump(
    obj: Any,
    destination: str | os.PathLike
) -> None:
    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    fd, temp_name = tempfile.mkstemp(
        prefix=(
            f".{destination.name}."
        ),
        suffix=".tmp",
        dir=destination.parent
    )

    try:
        with os.fdopen(
            fd,
            "wb"
        ) as f:
            pickle.dump(
                obj,
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

            f.flush()
            os.fsync(
                f.fileno()
            )

        os.replace(
            temp_name,
            destination
        )

    except Exception:
        try:
            os.unlink(
                temp_name
            )
        except FileNotFoundError:
            pass

        raise


## Step 2 — Test it in a temporary directory


In [ ]:
with tempfile.TemporaryDirectory() as folder:
    target = (
        Path(folder)
        / "checkpoint.pkl"
    )

    state = {
        "epoch": 12,
        "loss": 0.042,
        "finished": False,
    }

    atomic_pickle_dump(
        state,
        target
    )

    with target.open("rb") as f:
        restored_state = pickle.load(f)

    pprint(restored_state)


## Best Practice

Atomic replacement helps prevent partially written destination files.

It does not solve every durability problem, but it is much safer than overwriting the final file directly.


# Problem 14 — Reading Multiple Objects from One File

Pickle can store several objects one after another.

This can be useful for simple local streams or append-only records.


## Step 1 — Write several records


In [ ]:
stream = io.BytesIO()

for i in range(5):
    pickle.dump(
        {
            "sequence": i,
            "cube": i ** 3,
        },
        stream,
        protocol=pickle.HIGHEST_PROTOCOL
    )


## Step 2 — Read until `EOFError`

`EOFError` is the normal signal that there are no more pickle objects in the stream.


In [ ]:
stream.seek(0)

records = []

while True:
    try:
        records.append(
            pickle.load(stream)
        )
    except EOFError:
        break

pprint(records)


## Step 3 — Turn it into a reusable generator


In [ ]:
def iter_pickles(file_obj):
    while True:
        try:
            yield pickle.load(
                file_obj
            )
        except EOFError:
            return


In [ ]:
stream.seek(0)

for record in iter_pickles(
    stream
):
    print(record)


## Design Limitation

A raw sequence of pickles does not automatically give you:

- record lengths;
- checksums;
- indexes;
- easy corruption recovery;
- random access.

For robust storage formats, framing and metadata are often useful.


# Problem 15 — When Compression Helps

Pickle itself is not a general-purpose compression format.

If your data is repetitive, compressing the pickle may greatly reduce storage size.

But compression uses CPU.

We should measure both size and time.


## Step 1 — Create repetitive data


In [ ]:
events = [
    {
        "event": "page_view",
        "path": "/courses/python",
        "country": "BG",
    }
    for _ in range(10_000)
]


## Step 2 — Create the raw pickle


In [ ]:
raw_pickle = pickle.dumps(
    events,
    protocol=pickle.HIGHEST_PROTOCOL
)

len(raw_pickle)


## Step 3 — Compress with gzip


In [ ]:
compressed = gzip.compress(
    raw_pickle
)

len(compressed)


## Step 4 — Compute the compression ratio


In [ ]:
compression_ratio = (
    len(compressed)
    / len(raw_pickle)
)

compression_ratio


## Step 5 — Verify the round trip


In [ ]:
restored_events = pickle.loads(
    gzip.decompress(
        compressed
    )
)

restored_events == events


## Step 6 — Benchmark the trade-off


In [ ]:
def raw_round_trip():
    data = pickle.dumps(
        events,
        protocol=pickle.HIGHEST_PROTOCOL
    )
    return pickle.loads(data)


def gzip_round_trip():
    data = pickle.dumps(
        events,
        protocol=pickle.HIGHEST_PROTOCOL
    )

    compressed_data = gzip.compress(
        data
    )

    return pickle.loads(
        gzip.decompress(
            compressed_data
        )
    )


In [ ]:
raw_times = timeit.repeat(
    raw_round_trip,
    repeat=5,
    number=3
)

gzip_times = timeit.repeat(
    gzip_round_trip,
    repeat=5,
    number=3
)

print(
    "raw median:",
    statistics.median(raw_times)
)

print(
    "gzip median:",
    statistics.median(gzip_times)
)


## Solution Explanation

Compression may produce a much smaller file but a slower round trip.

The correct choice depends on what is scarce:

- disk space;
- network bandwidth;
- CPU time;
- latency.


# Problem 16 — Inspecting and Optimizing Pickle Bytecode

The `pickletools` module is useful for learning what pickle is actually doing.

It can also optimize some pickle streams.


## Step 1 — Serialize an object


In [ ]:
records = [
    {
        "id": i,
        "group": i % 5,
        "label": f"item-{i % 20}",
    }
    for i in range(1000)
]

original_pickle = pickle.dumps(
    records,
    protocol=pickle.HIGHEST_PROTOCOL
)


## Step 2 — Optimize the pickle


In [ ]:
optimized_pickle = pickletools.optimize(
    original_pickle
)

print(
    "original:",
    len(original_pickle)
)

print(
    "optimized:",
    len(optimized_pickle)
)


## Step 3 — Confirm it still represents the same value


In [ ]:
pickle.loads(
    optimized_pickle
) == records


## Step 4 — Inspect the opcodes

For large objects, the disassembly can be long.

We will use a smaller example here.


In [ ]:
small = {
    "x": [1, 2, 3],
    "y": [1, 2, 3],
}

small_pickle = pickle.dumps(
    small,
    protocol=pickle.HIGHEST_PROTOCOL
)

pickletools.dis(
    small_pickle
)


## Best Practice

`pickletools` is excellent for:

- learning;
- debugging;
- inspecting protocol behavior;
- studying memo operations.

But disassembly alone should not be treated as a complete security scanner.


# Problem 17 — Why Some Python Objects Pickle and Others Do Not

Pickle often serializes functions and classes **by reference**.

That means it records where the object can be imported from rather than serializing its source code.


## Step 1 — A top-level function


In [ ]:
def multiply_by_two(x):
    return x * 2


Let's see whether the function can be pickled.


In [ ]:
function_blob = pickle.dumps(
    multiply_by_two
)

restored_function = pickle.loads(
    function_blob
)

restored_function(10)


## Step 2 — A lambda


In [ ]:
anonymous_function = (
    lambda x: x * 2
)

try:
    pickle.dumps(
        anonymous_function
    )
except Exception as exc:
    print(
        type(exc).__name__,
        exc
    )


## Why the difference?

A normal top-level function has an importable module/name identity.

A lambda usually does not have a stable importable global name that pickle can resolve later.


## Step 3 — An open file handle


In [ ]:
with tempfile.TemporaryFile(
    mode="w+b"
) as f:

    try:
        pickle.dumps(f)
    except Exception as exc:
        print(
            type(exc).__name__,
            exc
        )


Open files represent active operating-system resources.

They are not ordinary reconstructible value objects.


## Best Practice

Before deciding to pickle a complex class, identify which fields represent:

- pure data;
- process-local resources;
- external resources;
- cached state;
- live connections.

Only pure, reconstructible state should usually be persisted.


# Problem 18 — Why Moving a Class Can Break Old Pickles

A pickle containing an instance of a class usually needs to find that class again when loading.

That makes long-lived pickle files dependent on your Python module structure.


## Conceptual Example

Suppose an object was originally created from:

```python
myapp.models.Customer
```

Later, you refactor the project and move the class to:

```python
myapp.domain.Customer
```

An older pickle may still refer to the old import path.

If that path no longer exists, loading can fail.


## Problem

What strategies reduce this fragility?

Think before reading the answer.


## Solution

Useful strategies include:

- keep serialized classes in stable modules;
- leave compatibility aliases/shims when moving important classes;
- version the stored state;
- write migration code;
- serialize plain dictionaries for very long-lived storage;
- use a language-neutral schema format when data must outlive application internals.

Pickle is convenient, but convenience is not the same thing as a durable archival schema.


# Problem 19 — Capstone: A Tutorial Checkpoint Format

We will combine several ideas into one design.

Our goal is a trusted-local checkpoint system that has:

- a format version;
- application state;
- HMAC authentication;
- atomic writes;
- version checks before returning state.

This is still a pickle-based format, so it is intended only for trusted environments.


## Step 1 — Define the checkpoint format version


In [ ]:
CHECKPOINT_VERSION = 1
CHECKPOINT_TAG_SIZE = (
    hashlib.sha256().digest_size
)


## Step 2 — Build the logical envelope

We do not serialize raw application state alone.

Instead, we wrap it in metadata.


In [ ]:
def make_envelope(state):
    return {
        "format_version": (
            CHECKPOINT_VERSION
        ),
        "state": state,
    }


## Step 3 — Serialize and authenticate


In [ ]:
def build_checkpoint_bytes(
    state,
    key: bytes
):
    envelope = make_envelope(
        state
    )

    payload = pickle.dumps(
        envelope,
        protocol=pickle.HIGHEST_PROTOCOL
    )

    tag = hmac.new(
        key,
        payload,
        hashlib.sha256
    ).digest()

    return tag + payload


## Step 4 — Verify before loading


In [ ]:
def parse_checkpoint_bytes(
    package: bytes,
    key: bytes
):
    if len(package) < CHECKPOINT_TAG_SIZE:
        raise ValueError(
            "Checkpoint is truncated"
        )

    received_tag = (
        package[:CHECKPOINT_TAG_SIZE]
    )

    payload = (
        package[CHECKPOINT_TAG_SIZE:]
    )

    expected_tag = hmac.new(
        key,
        payload,
        hashlib.sha256
    ).digest()

    if not hmac.compare_digest(
        received_tag,
        expected_tag
    ):
        raise ValueError(
            "Checkpoint authentication failed"
        )

    envelope = pickle.loads(
        payload
    )

    version = envelope.get(
        "format_version"
    )

    if version != CHECKPOINT_VERSION:
        raise ValueError(
            f"Unsupported checkpoint "
            f"version: {version!r}"
        )

    return envelope["state"]


## Step 5 — Add an atomic file writer


In [ ]:
def save_checkpoint(
    state,
    path,
    key: bytes
):
    package = build_checkpoint_bytes(
        state,
        key
    )

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    fd, temp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        suffix=".tmp",
        dir=path.parent
    )

    try:
        with os.fdopen(
            fd,
            "wb"
        ) as f:
            f.write(package)
            f.flush()
            os.fsync(
                f.fileno()
            )

        os.replace(
            temp_name,
            path
        )

    except Exception:
        try:
            os.unlink(
                temp_name
            )
        except FileNotFoundError:
            pass

        raise


## Step 6 — Add the file loader


In [ ]:
def load_checkpoint(
    path,
    key: bytes
):
    package = Path(
        path
    ).read_bytes()

    return parse_checkpoint_bytes(
        package,
        key
    )


## Step 7 — Test the complete system


In [ ]:
checkpoint_key = (
    b"tutorial-secret-key"
)

training_state = {
    "epoch": 9,
    "learning_rate": 0.001,
    "metrics": {
        "loss": 0.12,
        "accuracy": 0.96,
    },
}


In [ ]:
with tempfile.TemporaryDirectory() as folder:
    path = (
        Path(folder)
        / "training.chk"
    )

    save_checkpoint(
        training_state,
        path,
        checkpoint_key
    )

    restored_state = load_checkpoint(
        path,
        checkpoint_key
    )

    pprint(restored_state)


## Step 8 — Test tamper detection


In [ ]:
package = build_checkpoint_bytes(
    training_state,
    checkpoint_key
)

tampered = bytearray(package)
tampered[-3] ^= 1

try:
    parse_checkpoint_bytes(
        bytes(tampered),
        checkpoint_key
    )
except ValueError as exc:
    print(
        "Rejected:",
        exc
    )


## Capstone Review

This checkpoint format now demonstrates several real engineering ideas:

- pickle protocol selection;
- metadata envelopes;
- explicit format versions;
- authentication before deserialization;
- atomic replacement;
- clean separation between byte format and file I/O.

It is still not suitable for accepting arbitrary public input.

For untrusted data, use a safer serialization format.


# Final Concept Review

Try to answer each question before revealing the answer.

### Question 1

Why can two restored values be equal but not identical?


**Answer:** because unpickling reconstructs new Python objects. Equality compares value/state; identity asks whether two references point to the exact same object.


### Question 2

When does pickle preserve aliases?


**Answer:** when the repeated references are seen within the same pickle memo/object graph.


### Question 3

Why can separate pickle operations break shared-reference relationships?


**Answer:** each independent pickler starts with its own memo, so each operation reconstructs its own independent graph.


### Question 4

Why are cycles possible with pickle?


**Answer:** the memo lets pickle refer back to an object that was already encountered instead of recursively serializing forever.


### Question 5

Why should caches usually be excluded?


**Answer:** they are derived state, may become stale, and unnecessarily increase serialized size and coupling.


### Question 6

Why should `__setstate__` validate data?


**Answer:** restoring an object is another construction path. The class should re-establish its invariants instead of accepting invalid internal state.


### Question 7

Why version serialized state?


**Answer:** application classes evolve. Explicit versions make migrations intentional and testable.


### Question 8

Why verify HMAC before calling `pickle.loads`?


**Answer:** because deserialization is the dangerous step. Modified bytes must be rejected before the unpickler interprets them.


### Question 9

Why can moving a class break old pickle files?


**Answer:** pickle often stores references to classes/functions by module and qualified name. Refactoring import paths can invalidate those references.


### Question 10

When is pickle a good fit?


**Answer:** trusted Python-to-Python persistence where convenience, Python object fidelity, and local performance matter more than cross-language interoperability or untrusted-input safety.


# Final Best-Practices Checklist

Before using pickle in a real project, ask:

- Is the input trusted?
- Is the source authenticated?
- Do object identity relationships matter?
- Should connected objects be serialized together?
- Are transient fields excluded?
- Are invariants restored?
- Is the serialized state versioned?
- Are class import paths stable?
- Is the protocol compatible with every reader?
- Should files be written atomically?
- Would compression actually help?
- Would a language-neutral format be safer or more durable?

If any of these questions are unclear, that is a sign to design the persistence layer more carefully before relying on pickle.
